In [46]:

import pandas as pd


def answer_exact_match(pred, gold):
    return str(pred).strip() == str(gold).strip()


def program_exact_match(pred, gold):
    return str(pred).strip() == str(gold).strip()

def safe_exec(code: str):
    if code == "":
        return None
    local_vars = {}
    try:
        clean_code = code.strip().replace('\n ', '\n')
        exec(clean_code, {}, local_vars)
        return local_vars.get("answer", None)
    except Exception:
        try:
            clean_code2 = clean_code.split("```")[1].replace('python\n', '')
            exec(clean_code2, {}, local_vars)
            return local_vars.get("answer", None)
        except Exception:
            return None


def execution_accuracy(pred_answer, gold_answer, float_tol: float = 1e-3) -> bool:
    if pred_answer is None or gold_answer is None:
        return False
    if isinstance(pred_answer, (int, float)) and isinstance(gold_answer, (int, float)):
        return abs(pred_answer - gold_answer) < float_tol
    else:
        return str(pred_answer).strip() == str(gold_answer).strip()

In [ ]:
input_path = "../../results/test_cru_deepseek_coder_6_7b_instruct.json"
data = pd.read_json(input_path)
data['answer_exec'] = data['golden_program_generated'].apply(safe_exec)
data['generated_exec'] = data['generated_program'].apply(safe_exec)

<string>:3: SyntaxWarning: invalid escape sequence '\ '
<string>:3: SyntaxWarning: invalid escape sequence '\ '
<string>:3: SyntaxWarning: invalid escape sequence '\ '
<string>:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
<string>:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
<string>:3: SyntaxWarning: invalid escape sequence '\ '
<string>:12: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
<string>:16: Runtim

In [ ]:
data['execution_accuracy'] = data.apply(lambda row: execution_accuracy(
    row['generated_exec'], row['answer_exec']), axis = 1)
data['answer_exact_match'] = data.apply(lambda row: answer_exact_match(
    row['generated_exec'], row['answer']), axis = 1)
data['program_exact_match'] = data.apply(lambda row: program_exact_match(
    row['generated_program'], row['golden_program_generated']), axis = 1)

In [75]:
metrics = {
    'file': input_path,
    'execution_accuracy': data['execution_accuracy'].sum() / len(data),
    'answer_exact_match': data['answer_exact_match'].sum() / len(data),
    'program_exact_match': data['program_exact_match'].sum() / len(data)
}